In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

# Remonte jusqu'à trouver un dossier "data"
PROJECT_DIR = Path.cwd()
while not (PROJECT_DIR / "data").exists():
    if PROJECT_DIR.parent == PROJECT_DIR:
        raise FileNotFoundError("Je ne trouve pas 'data/'. Ouvre le dossier tripadvisor-ir-nlp dans VS Code.")
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
OUT_DIR = PROJECT_DIR / "outputs"
OUT_DIR.mkdir(exist_ok=True)

print("CWD        :", Path.cwd())
print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR   :", DATA_DIR)

CWD        : c:\Users\Jmaro\OneDrive\Documents\Esilv\Info retrieval and NLP\Projet\tripadvisor-ir-nlp\notebooks
PROJECT_DIR: c:\Users\Jmaro\OneDrive\Documents\Esilv\Info retrieval and NLP\Projet\tripadvisor-ir-nlp
DATA_DIR   : c:\Users\Jmaro\OneDrive\Documents\Esilv\Info retrieval and NLP\Projet\tripadvisor-ir-nlp\data


In [4]:
import os
from pathlib import Path

print("CWD =", os.getcwd())
print("data exists?", Path("data").exists())
print("files in data:", list(Path("data").glob("*"))[:10])

CWD = c:\Users\Jmaro\OneDrive\Documents\Esilv\Info retrieval and NLP\Projet\tripadvisor-ir-nlp\notebooks
data exists? False
files in data: []


In [5]:
reviews_path = DATA_DIR / "reviews83325.csv"
places_path  = DATA_DIR / "Tripadvisor.csv"

reviews = pd.read_csv(reviews_path)
places  = pd.read_csv(places_path, low_memory=False)

print("reviews shape:", reviews.shape)
print("places shape :", places.shape)

C:\Users\Jmaro\AppData\Local\Temp\ipykernel_19576\3673202792.py:4: DtypeWarning: Columns (0: owner_langue, 1: owner_date_review, 2: owner_connection, 3: owner_responder, 4: owner_response, 5: owner_title) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.read_csv(reviews_path)


reviews shape: (340385, 21)
places shape : (3761, 60)


In [6]:
print("reviews columns:", list(reviews.columns))
print("places columns :", list(places.columns))
reviews.head(2)

reviews columns: ['id', 'idplace', 'titre', 'idauteur', 'review', 'note', 'date_review', 'date_visit', 'langue', 'published_platform', 'typeReview', 'subratings', 'machine_translated', 'machine_translatable', 'owner_id', 'owner_langue', 'owner_date_review', 'owner_connection', 'owner_responder', 'owner_response', 'owner_title']
places columns : ['id', 'idTrip', 'fromId', 'nom', 'url', 'rating', 'nbAvis', 'nbAvisRecupere', 'latitude', 'longitude', 'typeR', 'adresse', 'priceRange', 'closed', 'hotelType', 'hotelStyle', 'hotelStars', 'hotelRoomNumber', 'hotelNoteEmplacement', 'hotelNoteProprete', 'hotelNoteService', 'HotelNoteQualitePrix', 'hoteldistance', 'hotelbearing', 'restaurantTypeCuisine', 'restaurantDietaryRestrictions', 'restaurantMeals', 'restaurantFeatures', 'restaurantNoteCuisine', 'restaurantNoteService', 'restaurantNoteQualitePrix', 'restaurantNoteAmbiance', 'activiteType', 'activiteSubCategorie', 'activiteSubType', 'website', 'nbScanReview', 'dateLastScanReviews', 'shape_gid

,id,idplace,titre,idauteur,review,note,date_review,date_visit,langue,published_platform,...,subratings,machine_translated,machine_translatable,owner_id,owner_langue,owner_date_review,owner_connection,owner_responder,owner_response,owner_title
0,771569620,188467,February,F645CC9429E8A40EB1F5A487780EC683,Personally I think it is the most beautiful sq...,5,23/9/2020 11:14:11,2020-02,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,769814072,188467,Nice green square,AFFB511F21DF819776CB2F8013034382,We walked through this lovely park but did not...,4,11/9/2020 07:52:32,2019-10,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# candidats possibles (on teste ce qui existe dans le CSV)
possible_text_cols = ["review", "text", "content", "comment", "Review", "Text"]
possible_lang_cols = ["langue", "language", "lang", "Langue", "Language"]
possible_id_cols   = ["idplace", "place_id", "id_place", "IdPlace"]

TEXT_COL = next((c for c in possible_text_cols if c in reviews.columns), None)
LANG_COL = next((c for c in possible_lang_cols if c in reviews.columns), None)
ID_COL   = next((c for c in possible_id_cols if c in reviews.columns), None)

print("Detected:", {"ID_COL": ID_COL, "TEXT_COL": TEXT_COL, "LANG_COL": LANG_COL})

Detected: {'ID_COL': 'idplace', 'TEXT_COL': 'review', 'LANG_COL': 'langue'}


In [8]:
reviews_en = reviews.copy()

# Filtre langue si on a la colonne
if LANG_COL is not None:
    reviews_en = reviews_en[reviews_en[LANG_COL].astype(str).str.lower().str.startswith("en")]

# Nettoyage minimal du texte
reviews_en = reviews_en.dropna(subset=[TEXT_COL, ID_COL])

print("reviews_en shape:", reviews_en.shape)
reviews_en[[ID_COL, TEXT_COL]].head(3)

reviews_en shape: (153071, 21)


,idplace,review
0,188467,Personally I think it is the most beautiful sq...
1,188467,We walked through this lovely park but did not...
2,188467,We come back to this huge square every time we...


In [9]:
MAX_REVIEWS_PER_PLACE = 20

# si on as une colonne de date, on peut trier; sinon, ça prendra l'ordre du fichier
date_candidates = ["date", "date_review", "review_date", "Date", "DateReview"]
DATE_COL = next((c for c in date_candidates if c in reviews_en.columns), None)

if DATE_COL:
    reviews_en = reviews_en.sort_values([ID_COL, DATE_COL])

reviews_cap = reviews_en.groupby(ID_COL).head(MAX_REVIEWS_PER_PLACE).copy()
print("reviews_cap shape:", reviews_cap.shape)

reviews_cap shape: (18447, 21)


In [10]:
place_texts = (
    reviews_cap.groupby(ID_COL)[TEXT_COL]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
    .rename(columns={ID_COL: "idplace", TEXT_COL: "text"})
)

print("place_texts shape:", place_texts.shape)
place_texts.head(3)

place_texts shape: (1835, 2)


,idplace,text
0,188467,We went there in November so it was not too wa...
1,188468,If you want to see the ancient Rome just walk ...
2,188470,"If you're looking for antique shops, walk into..."


In [11]:
# identifier la colonne id dans places
PLACE_ID_COL = "id" if "id" in places.columns else "ID"

# typeR est attendu dans l'énoncé
assert "typeR" in places.columns, "Colonne typeR introuvable dans Tripadvisor.csv"

place_texts = place_texts.merge(
    places[[PLACE_ID_COL, "typeR"]].rename(columns={PLACE_ID_COL: "idplace"}),
    on="idplace",
    how="inner"
)

print("place_texts shape after merge:", place_texts.shape)
print(place_texts["typeR"].value_counts(dropna=False))

place_texts shape after merge: (1835, 3)
typeR
AP    989
R     538
A     252
H      56
Name: count, dtype: int64


In [12]:
out_path = OUT_DIR / "place_texts_en_cap20.csv"
place_texts.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: c:\Users\Jmaro\OneDrive\Documents\Esilv\Info retrieval and NLP\Projet\tripadvisor-ir-nlp\outputs\place_texts_en_cap20.csv


In [13]:


rng = np.random.default_rng(42)

all_ids = place_texts["idplace"].unique()
rng.shuffle(all_ids)

mid = len(all_ids) // 2
train_ids = set(all_ids[:mid])   # queries
test_ids  = set(all_ids[mid:])   # candidates to retrieve

train_df = place_texts[place_texts["idplace"].isin(train_ids)].reset_index(drop=True)
test_df  = place_texts[place_texts["idplace"].isin(test_ids)].reset_index(drop=True)

print("train places:", train_df.shape)
print("test places :", test_df.shape)

train places: (917, 3)
test places : (918, 3)


In [14]:


def simple_tokenize(text: str):
    text = text.lower()
    # garder lettres + chiffres, remplacer le reste par espace
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = text.split()
    return tokens

In [15]:
from rank_bm25 import BM25Okapi

test_texts = test_df["text"].tolist()
test_tokens = [simple_tokenize(t) for t in test_texts]

bm25 = BM25Okapi(test_tokens)

In [22]:
def bm25_topk(query_text: str, top_k: int = 5):
    q = simple_tokenize(query_text)
    scores = bm25.get_scores(q)
    top_idx = np.argsort(scores)[::-1][:top_k]
    out = test_df.iloc[top_idx][["idplace", "typeR"]].copy()
    out["score"] = scores[top_idx]
    return out

bm25_topk(train_df.loc[0, "text"], top_k=5)

,idplace,typeR,score
279,7905549,A,1731.404875
5,191144,A,1699.973993
22,292257,A,1672.987504
31,641585,H,1611.690789
246,6642129,A,1597.268497


In [16]:
# Préparer arrays pour accélérer
test_type = test_df["typeR"].to_numpy()

def ranking_error_level1(query_text: str, query_typeR: str):
    q = simple_tokenize(query_text)
    scores = bm25.get_scores(q)
    ranked_idx = np.argsort(scores)[::-1]  # du meilleur au moins bon

    # trouver première position où typeR match
    match_positions = np.where(test_type[ranked_idx] == query_typeR)[0]
    if len(match_positions) == 0:
        return None  # undefined -> on ne teste pas cette query (comme dans l'énoncé)
    pos = int(match_positions[0])  # 0-based
    return pos  # erreur = pos (car "n-1" avec n position 1-based)

In [24]:
errs = []
for i in range(min(50, len(train_df))):
    e = ranking_error_level1(train_df.loc[i, "text"], train_df.loc[i, "typeR"])
    if e is not None:
        errs.append(e)

print("tested queries:", len(errs))
print("mean error:", sum(errs)/len(errs) if errs else None)
print("median error:", np.median(errs) if errs else None)

tested queries: 50
mean error: 1.3
median error: 0.0


In [17]:
# Colonnes utiles Level 2 (selon ton Tripadvisor.csv)
meta_cols = [
    "id", "typeR",
    "priceRange",
    "activiteSubCategorie", "activiteSubType",
    "restaurantTypeCuisine", "restaurantType",  # on gardera ce qu'on peut
]

# On garde seulement celles qui existent vraiment
meta_cols = [c for c in meta_cols if c in places.columns]

place_meta = places[meta_cols].rename(columns={"id": "idplace"}).copy()

place_texts2 = place_texts.drop(columns=[c for c in place_texts.columns if c in meta_cols and c != "idplace"], errors="ignore")
place_texts2 = place_texts2.merge(place_meta, on="idplace", how="left")

print(place_texts2.columns)
place_texts2.head(2)

Index(['idplace', 'text', 'typeR', 'priceRange', 'activiteSubCategorie',
       'activiteSubType', 'restaurantTypeCuisine', 'restaurantType'],
      dtype='str')


,idplace,text,typeR,priceRange,activiteSubCategorie,activiteSubType,restaurantTypeCuisine,restaurantType
0,188467,We went there in November so it was not too wa...,A,NaN,47,163,NaN,NaN
1,188468,If you want to see the ancient Rome just walk ...,A,NaN,47,163,NaN,NaN


In [18]:
rng = np.random.default_rng(42)
all_ids = place_texts2["idplace"].unique()
rng.shuffle(all_ids)

mid = len(all_ids) // 2
train_ids = set(all_ids[:mid])
test_ids  = set(all_ids[mid:])

train_df = place_texts2[place_texts2["idplace"].isin(train_ids)].reset_index(drop=True)
test_df  = place_texts2[place_texts2["idplace"].isin(test_ids)].reset_index(drop=True)

print(train_df.shape, test_df.shape)

(917, 8) (918, 8)


In [19]:
test_tokens = [simple_tokenize(t) for t in test_df["text"].tolist()]
bm25 = BM25Okapi(test_tokens)

In [20]:
def parse_id_list(x):
    """
    Retourne un set() d'entiers.
    x peut être NaN, int, float, str "1,2,3"
    """
    if pd.isna(x):
        return set()
    if isinstance(x, (int, np.integer)):
        return {int(x)}
    if isinstance(x, float):
        # parfois des ids stockés en float
        return {int(x)}
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return set()
    # split sur virgules
    parts = [p.strip() for p in s.split(",")]
    out = set()
    for p in parts:
        if p == "":
            continue
        # garder seulement les chiffres
        if p.isdigit():
            out.add(int(p))
        else:
            # si format bizarre, on essaye d'extraire les nombres
            nums = re.findall(r"\d+", p)
            out.update(int(n) for n in nums)
    return out

In [21]:
def match_level2(query_row, cand_row):
    t = query_row["typeR"]
    if pd.isna(t):
        return False
    
    # On exige le même type
    if cand_row["typeR"] != t:
        return False
    
    if t == "H":
        # match exact sur priceRange
        return (str(query_row.get("priceRange", "")).strip() != "" and
                query_row.get("priceRange") == cand_row.get("priceRange"))
    
    if t == "A":
        q_subtype = parse_id_list(query_row.get("activiteSubType"))
        c_subtype = parse_id_list(cand_row.get("activiteSubType"))
        if q_subtype and c_subtype and (q_subtype & c_subtype):
            return True
        # fallback subcategorie
        q_subcat = parse_id_list(query_row.get("activiteSubCategorie"))
        c_subcat = parse_id_list(cand_row.get("activiteSubCategorie"))
        return bool(q_subcat and c_subcat and (q_subcat & c_subcat))
    
    if t == "R":
        q_cui = parse_id_list(query_row.get("restaurantTypeCuisine"))
        c_cui = parse_id_list(cand_row.get("restaurantTypeCuisine"))
        return bool(q_cui and c_cui and (q_cui & c_cui))
    
    # AP ou autre : fallback type seulement
    return True

In [22]:
def ranking_error_level2(query_row):
    q = simple_tokenize(query_row["text"])
    scores = bm25.get_scores(q)
    ranked_idx = np.argsort(scores)[::-1]

    for pos, j in enumerate(ranked_idx):
        if match_level2(query_row, test_df.iloc[j]):
            return pos
    return None

In [31]:
errs2 = []
for i in range(min(50, len(train_df))):
    e = ranking_error_level2(train_df.iloc[i])
    if e is not None:
        errs2.append(e)

print("tested queries:", len(errs2))
print("mean error:", sum(errs2)/len(errs2) if errs2 else None)
print("median error:", np.median(errs2) if errs2 else None)

tested queries: 48
mean error: 6.666666666666667
median error: 1.0


In [23]:
def ranking_error_level1_from_row(query_row):
    q = simple_tokenize(query_row["text"])
    scores = bm25.get_scores(q)
    ranked_idx = np.argsort(scores)[::-1]
    qtype = query_row["typeR"]

    for pos, j in enumerate(ranked_idx):
        if test_df.iloc[j]["typeR"] == qtype:
            return pos
    return None

errs1_all, errs2_all = [], []
n_undef1, n_undef2 = 0, 0

for i in range(len(train_df)):
    e1 = ranking_error_level1_from_row(train_df.iloc[i])
    if e1 is None:
        n_undef1 += 1
    else:
        errs1_all.append(e1)

    e2 = ranking_error_level2(train_df.iloc[i])
    if e2 is None:
        n_undef2 += 1
    else:
        errs2_all.append(e2)

print("LEVEL 1 tested:", len(errs1_all), "undefined:", n_undef1)
print("LEVEL 1 mean  :", np.mean(errs1_all), "median:", np.median(errs1_all))

print("LEVEL 2 tested:", len(errs2_all), "undefined:", n_undef2)
print("LEVEL 2 mean  :", np.mean(errs2_all), "median:", np.median(errs2_all))

LEVEL 1 tested: 917 undefined: 0
LEVEL 1 mean  : 0.47546346782988 median: 0.0
LEVEL 2 tested: 882 undefined: 35
LEVEL 2 mean  : 2.4240362811791383 median: 0.0


In [24]:
results = {
    "level1_mean": float(np.mean(errs1_all)),
    "level1_median": float(np.median(errs1_all)),
    "level1_tested": len(errs1_all),
    "level1_undefined": n_undef1,
    "level2_mean": float(np.mean(errs2_all)),
    "level2_median": float(np.median(errs2_all)),
    "level2_tested": len(errs2_all),
    "level2_undefined": n_undef2,
}

results_df = pd.DataFrame([results])
results_df

,level1_mean,level1_median,level1_tested,level1_undefined,level2_mean,level2_median,level2_tested,level2_undefined
0,0.475463,0.0,917,0,2.424036,0.0,882,35


In [25]:
results_df.to_csv(OUT_DIR / "bm25_results.csv", index=False)
print("Saved results to outputs/bm25_results.csv")

Saved results to outputs/bm25_results.csv


In [26]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

# Encoder test corpus
test_embeddings = model.encode(test_df["text"].tolist(), show_progress_bar=True)

print("Embeddings shape:", test_embeddings.shape)

c:\Users\Jmaro\OneDrive\Documents\Esilv\Info retrieval and NLP\Projet\tripadvisor-ir-nlp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 140.96it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 29/29 [01:56<00:00,  4.00s/it]

Embeddings shape: (918, 384)


In [27]:
def embed_topk(query_text, top_k=5):
    q_emb = model.encode([query_text])
    sims = cosine_similarity(q_emb, test_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    
    out = test_df.iloc[top_idx][["idplace", "typeR"]].copy()
    out["score"] = sims[top_idx]
    return out

In [28]:
embed_topk(train_df.loc[0, "text"], 5)

,idplace,typeR,score
22,292257,A,0.694004
246,6642129,A,0.686941
279,7905549,A,0.678631
210,4776710,H,0.661506
592,13165255,R,0.657205


In [29]:
def ranking_error_level2_embed(query_row):
    q_emb = model.encode([query_row["text"]])
    sims = cosine_similarity(q_emb, test_embeddings)[0]
    ranked_idx = np.argsort(sims)[::-1]

    for pos, j in enumerate(ranked_idx):
        if match_level2(query_row, test_df.iloc[j]):
            return pos
    return None

In [40]:
errs2_embed = []
for i in range(min(50, len(train_df))):
    e = ranking_error_level2_embed(train_df.iloc[i])
    if e is not None:
        errs2_embed.append(e)

print("mean:", np.mean(errs2_embed))
print("median:", np.median(errs2_embed))

mean: 7.25
median: 1.0
